# SDQC with a hardware-measured reference (V5.1)

**What changed from V5, and why.** In V5, SDQC compared today's probe scores with *noise-free simulator* scores. That has a logical problem: if you can simulate the model without noise, you could run the whole model on the simulator and would not need quantum hardware. This version removes that dependency.

**SDQC (this version) uses only quantities that can be measured on hardware:**
1. **Calibration day.** The hardware is in its reference state. Fit the Platt calibrator on labeled calibration images (as Fixed Platt does), and measure a fixed set of unlabeled **probe images** on the hardware with a high shot count. Store those probe scores.
2. **Deployment day.** Re-run the *same* probes on today's hardware, fit an affine map between today's and calibration-day probe scores, use it to map today's patient scores back to the calibration-day scale, apply the calibration-day Platt calibrator, then re-estimate the disease rate with EM.

No noise-free simulation, no reported noise rates and no new patient labels are needed. The V5 simulator-referenced variant is kept as **SDQC-sim** so the cost of removing the simulator can be measured.

**Other changes:** an oracle that knows both the noise *and* the true disease rate (V5's oracle adjusted only for noise); OCTMNIST pool subsampled by default to keep run time reasonable; summary tables with paired bootstrap confidence intervals and a figure for the hard case (unreported noise + changed disease rate).

**Datasets:** PneumoniaMNIST, BreastMNIST (small; exploratory), and a **derived binary OCTMNIST task: CNV (class 0) vs NORMAL (class 3)**, other classes excluded.

**Run in Colab:** `Runtime → Change runtime type → T4 GPU` (feature extraction for OCTMNIST took over 2 hours on CPU in V5), then `Runtime → Run all`. Set `QUICK_TEST = True` first for a short check.

In [ ]:
!pip install -q pennylane medmnist

## 1. Configuration and imports

In [ ]:
import os, time, warnings, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision
import pennylane as qml
import medmnist
from medmnist import INFO
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.svm import SVC
from scipy.optimize import minimize_scalar, minimize

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

QUICK_TEST = False

CFG = dict(
    DATASETS=["pneumoniamnist", "breastmnist", "octmnist"],
    IMG_SIZE=64,
    OCT_MAX_POOL=15000,          # OCTMNIST train+val images kept (stratified) before feature extraction; None = all ~108k
    TEST_FRAC=0.15, CAL_FRAC=0.15, VAL_FRAC=0.10,
    N_QUBITS=4, N_LAYERS=3,
    EPOCHS=40, BATCH=64, LR=0.02, PATIENCE=8,
    SEEDS=[0, 1, 2],                       # use 5-10 for the paper
    QSVM_N_TRAIN=600, QSVM_C=1.0,
    AUROC_GATE=0.75,
    # Hardware drift (same design as V4/V5)
    FIT_P=[0.0, 0.002, 0.01, 0.03],
    HELDOUT_P=[0.001, 0.005, 0.015, 0.02, 0.05],
    UNSEEN_AMP=[0.01, 0.03], UNSEEN_BASE_P=0.002,
    N_READOUT_TRAIN=6, N_READOUT_TEST=4, READOUT_MAX=0.08, DRIFT_SHOTS=1000,
    REF=dict(p=0.002, g=0.0, e0=0.02, e1=0.02),     # calibration-day hardware state
    REPORT_ERROR=0.2, COND_SEED=123,
    PRIORS=["source", 0.30, 0.50, 0.90],
    # SDQC
    PROBE_N=96,                  # number of fixed probe images
    REF_PROBE_SHOTS=10000,       # calibration-day probe measurement (one-time cost; reduces noise in the reference)
    SDQC_MIN_SLOPE=0.10, SDQC_MAX_SLOPE=5.0,
    SDQC_EM_SHRINK=0.85,         # prespecified; not tuned on test labels
    N_BINS=15, N_BOOT=2000,
)
if QUICK_TEST:
    CFG.update(DATASETS=["pneumoniamnist", "octmnist"], OCT_MAX_POOL=3000, EPOCHS=5, SEEDS=[0], QSVM_N_TRAIN=150,
               FIT_P=[0.0, 0.002, 0.03], HELDOUT_P=[0.005, 0.05], UNSEEN_AMP=[0.03],
               N_READOUT_TRAIN=3, N_READOUT_TEST=2, N_BOOT=200)
# Part A settings kept so the shared V4 cells still run
CFG.update(A_DEPOL=[0.02], A_AMP=[0.02], A_SHOTS=[1000], A_READOUT=[(0.01, 0.08)],
           JOINT=dict(p=0.015, g=0.0, e0=0.02, e1=0.05), BATCH_SIZES=[64])

NQ, NL = CFG["N_QUBITS"], CFG["N_LAYERS"]
torch.set_default_dtype(torch.float64)
CACHE = "/content/v5_cache" if os.path.isdir("/content") else "v5_cache"
os.makedirs(CACHE, exist_ok=True)
print("PennyLane", qml.__version__, "| torch", torch.__version__, "| medmnist", medmnist.__version__)

## 2. Image features and data splits

Frozen ImageNet ResNet-18 features; standardization and PCA are fitted on training data only. For OCTMNIST only **CNV = 0** and **NORMAL = 3** are kept (CNV = positive class). Because OCTMNIST is large, the pooled train+val images are stratified-subsampled to `OCT_MAX_POOL` before feature extraction; the official test set is kept in full as the shift set. Features are cached, so re-running is fast.

In [ ]:
def load_split(flag, split):
    DataClass = getattr(medmnist, INFO[flag]["python_class"])
    try:
        ds = DataClass(split=split, download=True, size=CFG["IMG_SIZE"])
    except TypeError:
        ds = DataClass(split=split, download=True)
    labels = ds.labels.ravel().astype(int)
    if flag == "octmnist":
        keep = np.isin(labels, [0, 3])
        return ds.imgs[keep], (labels[keep] == 0).astype(int)  # CNV=1, NORMAL=0
    return ds.imgs, labels

device = "cuda" if torch.cuda.is_available() else "cpu"
encoder = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
encoder.fc = nn.Identity()
encoder = encoder.float().eval().to(device)
MEAN = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32).view(1, 3, 1, 1)

@torch.no_grad()
def extract_features(imgs, bs=128):
    out = []
    for i in range(0, len(imgs), bs):
        x = torch.from_numpy(imgs[i:i + bs]).float() / 255.0
        x = x.unsqueeze(1) if x.ndim == 3 else x.permute(0, 3, 1, 2)
        if x.shape[1] == 1:
            x = x.repeat(1, 3, 1, 1)
        x = nn.functional.interpolate(x, size=224, mode="bilinear", align_corners=False)
        out.append(encoder(((x - MEAN) / STD).to(device)).cpu().numpy())
    return np.concatenate(out).astype(np.float64)

DATA = {}
for flag in CFG["DATASETS"]:
    cap = CFG["OCT_MAX_POOL"] if flag == "octmnist" else None
    path = f"{CACHE}/{flag}_{CFG['IMG_SIZE']}_binaryv5_{cap or 'all'}.npz"
    if os.path.exists(path):
        d = np.load(path); F_pool, y_pool, F_shift, y_shift = d["Fp"], d["yp"], d["Fs"], d["ys"]
    else:
        t0 = time.time()
        Xtr, ytr = load_split(flag, "train"); Xva, yva = load_split(flag, "val"); Xte, yte = load_split(flag, "test")
        Xp, yp = np.concatenate([Xtr, Xva]), np.concatenate([ytr, yva])
        if cap and len(yp) > cap:
            keep = train_test_split(np.arange(len(yp)), train_size=cap, stratify=yp, random_state=0)[0]
            Xp, yp = Xp[keep], yp[keep]
        F_pool, y_pool = extract_features(Xp), yp
        F_shift, y_shift = extract_features(Xte), yte
        np.savez(path, Fp=F_pool, yp=y_pool, Fs=F_shift, ys=y_shift)
        print(f"{flag}: features in {time.time() - t0:.0f}s on {device}")
    idx = np.arange(len(y_pool))
    r_, i_test = train_test_split(idx, test_size=CFG["TEST_FRAC"], stratify=y_pool, random_state=0)
    r_, i_cal = train_test_split(r_, test_size=CFG["CAL_FRAC"] / (1 - CFG["TEST_FRAC"]), stratify=y_pool[r_], random_state=0)
    i_tr, i_val = train_test_split(r_, test_size=CFG["VAL_FRAC"] / (1 - CFG["TEST_FRAC"] - CFG["CAL_FRAC"]),
                                   stratify=y_pool[r_], random_state=0)
    if QUICK_TEST:
        i_tr = i_tr[:600]
    F = {"train": F_pool[i_tr], "val": F_pool[i_val], "cal": F_pool[i_cal], "test": F_pool[i_test], "shift": F_shift}
    Y = {"train": y_pool[i_tr], "val": y_pool[i_val], "cal": y_pool[i_cal], "test": y_pool[i_test], "shift": y_shift}
    sc1 = StandardScaler().fit(F["train"])
    pca = PCA(n_components=NQ, random_state=0).fit(sc1.transform(F["train"]))
    sc2 = StandardScaler().fit(pca.transform(sc1.transform(F["train"])))
    Z = {k: sc2.transform(pca.transform(sc1.transform(v))) for k, v in F.items()}
    A = {k: np.clip(v, -3, 3) * (np.pi / 6) + np.pi / 2 for k, v in Z.items()}     # [0, pi]: no wrap-around
    DATA[flag] = dict(Z=Z, A=A, Y=Y, labels=("CNV=1, NORMAL=0 (derived binary)" if flag == "octmnist" else INFO[flag]["label"]))
    print(f"\n{flag}  labels={DATA[flag]['labels']}  PCA variance kept={pca.explained_variance_ratio_.sum():.2f}")
    for k in Y:
        print(f"   {k:5s} n={len(Y[k]):5d}  positive share={Y[k].mean():.2f}")

## 3. Quantum circuits and noise channels (from corrected V4)

In [ ]:
def _noise(w, p, g):
    if p > 0:
        qml.DepolarizingChannel(p, wires=w)
    if g > 0:
        qml.AmplitudeDamping(g, wires=w)

def vqc_circuit(x, W, p=0.0, g=0.0, reupload=False):
    for l in range(NL):
        if l == 0 or reupload:
            for q in range(NQ):
                qml.RY(x[..., q], wires=q); _noise(q, p, g)
        for q in range(NQ):
            qml.Rot(W[l, q, 0], W[l, q, 1], W[l, q, 2], wires=q); _noise(q, p, g)
        for q in range(NQ):
            t = (q + 1) % NQ
            qml.CNOT(wires=[q, t]); _noise(q, p, g); _noise(t, p, g)
    return qml.expval(qml.PauliZ(0))

def feature_map(x, p=0.0, g=0.0):
    for _ in range(2):
        for q in range(NQ):
            qml.RY(x[..., q], wires=q); _noise(q, p, g)
            qml.RZ(x[..., q], wires=q); _noise(q, p, g)
        for q in range(NQ - 1):
            qml.CNOT(wires=[q, q + 1]); _noise(q, p, g); _noise(q + 1, p, g)

dev_ideal = qml.device("default.qubit", wires=NQ)
dev_mixed = qml.device("default.mixed", wires=NQ)
qn_vqc_train = qml.QNode(vqc_circuit, dev_ideal, interface="torch", diff_method="backprop")
qn_vqc_noisy = qml.QNode(vqc_circuit, dev_mixed, interface="numpy")

@qml.qnode(dev_ideal, interface="numpy")
def qn_state(x):
    feature_map(x)
    return qml.state()

@qml.qnode(dev_mixed, interface="numpy")
def qn_dm(x, p, g):
    feature_map(x, p, g)
    return qml.density_matrix(wires=range(NQ))

def batched(fn, X, *args):
    try:
        return np.asarray(fn(X, *args))
    except Exception:
        return np.stack([np.asarray(fn(x, *args)) for x in X])

def apply_readout(f, e0, e1):
    return f * (1 - e0 - e1) + (e1 - e0)

def apply_shots_expval(f, shots, rng):
    return 1 - 2 * rng.binomial(shots, np.clip((1 - f) / 2, 0, 1)) / shots

def cond(p=0.0, g=0.0, e0=0.0, e1=0.0, shots=None):
    return dict(p=p, g=g, e0=e0, e1=e1, shots=shots)
CLEAN = cond()

## 4. Quantum and classical models (from corrected V4)

In [ ]:
def train_torch(model, Xtr, ytr, Xva, yva, seed):
    torch.manual_seed(seed)
    Xtr, ytr = torch.tensor(Xtr), torch.tensor(ytr, dtype=torch.float64)
    Xva, yva = torch.tensor(Xva), torch.tensor(yva, dtype=torch.float64)
    opt = torch.optim.Adam(model.parameters(), lr=CFG["LR"])
    lossf = nn.BCEWithLogitsLoss()
    best, state, bad = np.inf, None, 0
    for ep in range(CFG["EPOCHS"]):
        perm = torch.randperm(len(ytr))
        for i in range(0, len(ytr), CFG["BATCH"]):
            j = perm[i:i + CFG["BATCH"]]
            opt.zero_grad(); lossf(model(Xtr[j]), ytr[j]).backward(); opt.step()
        with torch.no_grad():
            vl = lossf(model(Xva), yva).item()
        if vl < best - 1e-4:
            best, bad, state = vl, 0, {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= CFG["PATIENCE"]:
                break
    model.load_state_dict(state)
    return model, best, ep + 1

class VQCNet(nn.Module):
    def __init__(self, seed, reupload):
        super().__init__()
        g = torch.Generator().manual_seed(seed)
        self.W = nn.Parameter(torch.rand(NL, NQ, 3, generator=g) * 2 * np.pi)
        self.a = nn.Parameter(torch.tensor(2.0)); self.b = nn.Parameter(torch.tensor(0.0))
        self.reupload = reupload
    def forward(self, x):
        return self.a * qn_vqc_train(x, self.W, 0.0, 0.0, self.reupload) + self.b

class VQCModel:
    kind = "quantum"
    def __init__(self, name, D, seed, reupload):
        self.name, self.D, self.reupload = name, D, reupload
        net, self.val_loss, self.epochs = train_torch(VQCNet(seed, reupload), D["A"]["train"], D["Y"]["train"],
                                                      D["A"]["val"], D["Y"]["val"], seed)
        self.W = net.W.detach().numpy(); self.a, self.b = net.a.item(), net.b.item()
        with torch.no_grad():
            self._f = {(s, 0.0, 0.0): qn_vqc_train(torch.tensor(D["A"][s]), net.W, 0.0, 0.0, reupload).detach().numpy()
                       for s in ["val", "cal", "test", "shift"]}
    def exact(self, split, p, g):
        key = (split, p, g)
        if key not in self._f:
            self._f[key] = batched(lambda X, *a: qn_vqc_noisy(X, self.W, p, g, self.reupload), self.D["A"][split]).astype(float)
        return self._f[key]
    def clean(self, split):
        return self.a * self.exact(split, 0.0, 0.0) + self.b
    def observe(self, split, c, rng):
        f = self.exact(split, c["p"], c["g"])
        if c["e0"] or c["e1"]:
            f = apply_readout(f, c["e0"], c["e1"])
        if c["shots"]:
            f = apply_shots_expval(f, c["shots"], rng)
        return self.a * f + self.b

class QSVMModel:
    kind = "quantum"
    def __init__(self, name, D, seed):
        self.name, self.D = name, D
        Ytr = D["Y"]["train"]
        n = min(CFG["QSVM_N_TRAIN"], len(Ytr))
        self.idx = train_test_split(np.arange(len(Ytr)), train_size=n, stratify=Ytr, random_state=seed)[0] if n < len(Ytr) else np.arange(len(Ytr))
        self.Xtr = D["A"]["train"][self.idx]
        self._sv = {"train": batched(qn_state, self.Xtr)}
        for s in ["val", "cal", "test", "shift"]:
            self._sv[s] = batched(qn_state, D["A"][s])
        K = np.abs(self._sv["train"].conj() @ self._sv["train"].T) ** 2
        self.clf = SVC(C=CFG["QSVM_C"], kernel="precomputed").fit(K, Ytr[self.idx])
        self._dm = {}
    def _dms(self, split, p, g):
        key = (split, p, g)
        if key not in self._dm:
            X = self.Xtr if split == "train" else self.D["A"][split]
            self._dm[key] = batched(qn_dm, X, p, g).reshape(len(X), -1)
        return self._dm[key]

    def kernel(self, split, p, g):
        if p == 0 and g == 0:
            K = np.abs(self._sv[split].conj() @ self._sv["train"].T) ** 2
        else:
            K = np.real(self._dms(split, p, g) @ self._dms("train", p, g).conj().T)
        return np.clip(np.nan_to_num(K), 0.0, 1.0)      # rounding can give 1.0000000000000002
    def clean(self, split):
        return self.clf.decision_function(self.kernel(split, 0.0, 0.0))
    def observe(self, split, c, rng):
        K = self.kernel(split, c["p"], c["g"])          # readout error not modelled for the kernel proxy
        if c["shots"]:
            K = rng.binomial(c["shots"], K) / c["shots"]
        return self.clf.decision_function(K)

class ClassicalModel:
    kind = "classical"
    def __init__(self, name, D, seed):
        self.name, self.D = name, D
        if name == "MLP":
            torch.manual_seed(seed)
            net = nn.Sequential(nn.Linear(NQ, 6), nn.Tanh(), nn.Linear(6, 1), nn.Flatten(0))
            net, self.val_loss, self.epochs = train_torch(net, D["Z"]["train"], D["Y"]["train"], D["Z"]["val"], D["Y"]["val"], seed)
            with torch.no_grad():
                self._z = {s: net(torch.tensor(D["Z"][s])).numpy() for s in ["val", "cal", "test", "shift"]}
        else:
            svm = SVC(C=1.0, kernel="rbf", gamma="scale", random_state=seed).fit(D["Z"]["train"], D["Y"]["train"])
            self._z = {s: svm.decision_function(D["Z"][s]) for s in ["val", "cal", "test", "shift"]}
    def clean(self, split):
        return self._z[split]
    def observe(self, split, c, rng):
        return self._z[split]

class ControlModel:
    """Classical MLP whose logits receive the same stretch, shift and noise that the VQC's logits
    received under the same condition. If it behaves like the VQC, the effects come from the
    noise physics, not from the quantum classifier itself."""
    kind = "control"
    def __init__(self, name, mlp, vqc):
        self.name, self.mlp, self.vqc = name, mlp, vqc
    def clean(self, split):
        return self.mlp.clean(split)
    def observe(self, split, c, rng):
        zq = self.vqc.observe(split, c, rng); z0 = self.vqc.clean(split)
        s, off = np.polyfit(z0, zq, 1)
        sd = np.std(zq - (s * z0 + off))
        return s * self.mlp.clean(split) + off + rng.normal(0, sd, len(z0))

## 5. Calibration methods and metrics (from corrected V4)

In [ ]:
sigmoid = lambda z: 1 / (1 + np.exp(-np.clip(z, -50, 50)))
nll_logits = lambda z, y: np.mean(np.logaddexp(0, z) - y * z)

def ece_score(p, y, n_bins=CFG["N_BINS"]):
    b = np.minimum((p * n_bins).astype(int), n_bins - 1)
    return sum((b == k).mean() * abs(p[b == k].mean() - y[b == k].mean()) for k in range(n_bins) if (b == k).any())

def metrics(p, y):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return dict(Brier=np.mean((p - y) ** 2), NLL=-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)),
                ECE=ece_score(p, y), AUROC=roc_auc_score(y, p) if 0 < y.mean() < 1 else np.nan,
                Acc=np.mean((p >= 0.5) == y), CITL=p.mean() - y.mean(), n=len(y))

def fit_platt(z, y):
    a_, b_ = minimize(lambda v: nll_logits(v[0] * z + v[1], y), x0=[1.0, 0.0], method="BFGS").x
    return lambda zz: sigmoid(a_ * zz + b_)

def fit_temperature(z, y):
    T = np.exp(minimize_scalar(lambda lt: nll_logits(z / np.exp(lt), y), bounds=(-6, 6), method="bounded").x)
    return lambda zz: sigmoid(zz / T)

def em_prior(p, pi_s, iters=200):
    """Saerens et al. (2002): re-estimate the class prior on unlabeled data and adjust probabilities."""
    p = np.clip(p, 1e-6, 1 - 1e-6); pi_t = pi_s
    for _ in range(iters):
        num = pi_t / pi_s * p; q = num / (num + (1 - pi_t) / (1 - pi_s) * (1 - p))
        new = float(np.clip(q.mean(), 0.01, 0.99))
        if abs(new - pi_t) < 1e-7:
            break
        pi_t = new
    return q

def prior_adjust(p, pi_s, pi_t):
    p = np.clip(p, 1e-6, 1 - 1e-6); num = pi_t / pi_s * p
    return num / (num + (1 - pi_t) / (1 - pi_s) * (1 - p))

def eta(c):
    return np.array([c["p"], c["e0"], c["e1"]])

def fit_noise_conditioned_platt(zs, cs, y):
    raw = np.array([eta(c) for c in cs]); mu, sd = raw.mean(0), raw.std(0) + 1e-12
    Phi = np.vstack([np.repeat(((eta(c) - mu) / sd)[None], len(z), 0) for z, c in zip(zs, cs)])
    zz, yy = np.concatenate(zs), np.tile(y, len(zs))
    w = minimize(lambda w: nll_logits((w[0] + Phi @ w[1:4]) * zz + w[4] + Phi @ w[5:8], yy),
                 x0=np.r_[1.0, 0, 0, 0, 0.0, 0, 0, 0], method="BFGS").x
    def predict(z, c):
        ph = (eta(c) - mu) / sd
        return sigmoid((w[0] + ph @ w[1:4]) * z + w[4] + ph @ w[5:8])
    return predict

def fit_device_response(z0, zs, cs):
    """Physics-informed: score_obs ~= s(eta)*score_clean + c(eta), with
    log s = k*log(1-4p/3) + m*log(1-e0-e1)  and  c = c2*(1-s) + c1*(e1-e0).
    Fitted on UNLABELED calibration images (paired noise-free vs noisy scores) at the training conditions."""
    S, O = [], []
    for z in zs:
        s, o = np.polyfit(z0, z, 1); S.append(max(s, 1e-4)); O.append(o)
    S, O = np.array(S), np.array(O)
    X1 = np.array([[np.log(1 - 4 * c["p"] / 3), np.log(1 - c["e0"] - c["e1"])] for c in cs])
    k, m = np.linalg.lstsq(X1, np.log(S), rcond=None)[0]
    X2 = np.column_stack([1 - S, [c["e1"] - c["e0"] for c in cs]])
    c2, c1 = np.linalg.lstsq(X2, O, rcond=None)[0]
    def correct(z, c):
        s = np.exp(k * np.log(max(1 - 4 * c["p"] / 3, 1e-6)) + m * np.log(max(1 - c["e0"] - c["e1"], 1e-6)))
        s = max(s, 0.02)
        return (z - (c2 * (1 - s) + c1 * (c["e1"] - c["e0"]))) / s
    return correct, dict(k=k, m=m, c1=c1, c2=c2)

def moment_match(z, ref_mean, ref_std):
    return (z - z.mean()) / (z.std() + 1e-12) * ref_std + ref_mean

## 6. Hardware drift profiles (from corrected V4)

In [ ]:
rc = np.random.default_rng(CFG["COND_SEED"])
EM_, SH = CFG["READOUT_MAX"], CFG["DRIFT_SHOTS"]
TRAIN_CONDS = [cond(p, 0.0, rc.uniform(0, EM_), rc.uniform(0, EM_), SH) for p in CFG["FIT_P"] for _ in range(CFG["N_READOUT_TRAIN"])]
TEST_CONDS = [("held-out depolarizing", cond(p, 0.0, rc.uniform(0, EM_), rc.uniform(0, EM_), SH))
              for p in CFG["HELDOUT_P"] for _ in range(CFG["N_READOUT_TEST"])]
TEST_CONDS += [("unseen: amplitude damping", cond(CFG["UNSEEN_BASE_P"], gm, rc.uniform(0, EM_), rc.uniform(0, EM_), SH))
               for gm in CFG["UNSEEN_AMP"] for _ in range(CFG["N_READOUT_TEST"])]
def reported(c):
    f = lambda v: v * np.exp(rc.normal(0, CFG["REPORT_ERROR"]))
    return cond(f(c["p"]), 0.0, f(c["e0"]), f(c["e1"]), c["shots"])     # amplitude damping is never reported
TEST_REPORTED = [reported(c) for _, c in TEST_CONDS]
REF = cond(**CFG["REF"], shots=SH)
COND_A = [("clean", "0", CLEAN)]
COND_A += [("depolarizing", str(p), cond(p=p)) for p in CFG["A_DEPOL"]]
COND_A += [("amplitude damping", str(gm), cond(g=gm)) for gm in CFG["A_AMP"]]
COND_A += [("shots", str(S), cond(shots=S)) for S in CFG["A_SHOTS"]]
COND_A += [("readout", f"{e0}/{e1}", cond(e0=e0, e1=e1)) for e0, e1 in CFG["A_READOUT"]]
COND_A += [("combined", "realistic", cond(0.005, 0.005, 0.01, 0.05, 1000))]
JOINT = cond(**CFG["JOINT"], shots=SH)
print(f"Part B: {len(TRAIN_CONDS)} training conditions, {len(TEST_CONDS)} test conditions")

## 7. SDQC with a hardware-measured reference

**Calibration day (hardware in its reference state, labels available).**
- Fit Platt scaling on the labeled calibration set's scores → the calibration-day calibrator (identical to Fixed Platt).
- Choose `PROBE_N` fixed probe images (inputs from the validation split; their labels are never used) and measure their scores on the hardware with `REF_PROBE_SHOTS` shots. Store these reference probe scores.

**Deployment (hardware may have drifted; no new labels).**
- **Step 1, hardware correction:** re-run the same probes; fit `today ≈ a · reference + b` by least squares; map each patient score `z` back to the calibration-day scale, `(z − b) / a` (slope bounded for stability).
- **Step 2, calibration:** apply the calibration-day Platt calibrator.
- **Step 3, population correction:** re-estimate the disease rate on the unlabeled batch with EM (Saerens et al., 2002) and adjust the probabilities.

Because the probes are the same images every time, a change in the patient population cannot change the probe scores; only the hardware can. That is what lets SDQC separate the two kinds of change.

**Comparators.** `SDQC-sim` is V5's version (reference = noise-free simulator scores; noise-free Platt calibrator). The oracle `Oracle (noise + true prior)` is refitted with labels under today's noise and adjusted to the true disease rate; it is a best case, not a usable method.

**Assumptions to state in the paper.** The hardware distortion seen by the probes must transfer to patient images (true for the input-independent noise simulated here); EM assumes label shift (class-conditional score distributions unchanged).

In [ ]:
def select_probe_indices(n, count, seed=31415):
    return np.sort(np.random.default_rng(seed).choice(n, size=min(n, count), replace=False))


def fit_probe_response(z_reference, z_today, min_slope=0.10, max_slope=5.0):
    """No labels. Fit z_today ~= slope * z_reference + intercept on the fixed probes and return
    a function that maps today's scores back to the reference scale."""
    x, y = np.asarray(z_reference, float), np.asarray(z_today, float)
    if len(x) < 3 or np.std(x) < 1e-8:
        return (lambda z: np.asarray(z)), dict(slope=1., intercept=0., probe_rmse=np.nan, status="degenerate")
    slope, intercept = np.polyfit(x, y, 1)
    if not np.isfinite(slope) or slope <= 0:
        return (lambda z: np.asarray(z)), dict(slope=float(slope), intercept=float(intercept), probe_rmse=np.nan, status="nonpositive_slope")
    slope = float(np.clip(slope, min_slope, max_slope))
    intercept = float(np.mean(y - slope * x))
    rmse = float(np.sqrt(np.mean((y - (slope * x + intercept)) ** 2)))
    def correct(z):
        return (np.asarray(z, float) - intercept) / slope
    return correct, dict(slope=slope, intercept=intercept, probe_rmse=rmse, status="ok")


def em_prior_with_info(p_source, source_prior, shrink=1., max_iters=200):
    """Estimate the target disease rate from UNLABELED probabilities (EM), then adjust each probability."""
    p = np.clip(np.asarray(p_source, float), 1e-6, 1 - 1e-6)
    pi_s = float(np.clip(source_prior, 1e-4, 1 - 1e-4))
    pi = pi_s
    for _ in range(max_iters):
        q = prior_adjust(p, pi_s, pi)
        updated = float(np.clip(q.mean(), 0.01, 0.99))
        if abs(updated - pi) < 1e-7:
            break
        pi = updated
    pi_est = float(pi_s + shrink * (pi - pi_s))
    return prior_adjust(p, pi_s, pi_est), pi_est


def sample_label_shift(y, target_prior, seed):
    """Without replacement; labels only set the controlled composition of the evaluation subset."""
    r = np.random.default_rng(seed)
    pos, neg = np.flatnonzero(y == 1), np.flatnonzero(y == 0)
    if target_prior == "source":
        return np.arange(len(y))
    pi = float(target_prior)
    n = int(min(len(y), len(pos) / pi, len(neg) / (1 - pi)))
    k = int(round(n * pi))
    if n < 20 or k < 5 or n - k < 5:
        return np.array([], dtype=int)
    return np.sort(np.r_[r.choice(pos, k, replace=False), r.choice(neg, n - k, replace=False)])

## 8. Run the benchmark

Hardware conditions (no noise, 20 held-out depolarizing, 8 unseen amplitude damping) are crossed with disease rates (source, 30%, 50%, 90%). Every method receives identical noisy patient scores for each condition. Methods marked *sim* or *physics-informed* use noise-free simulator outputs and are reported for comparison only.

In [ ]:
from pathlib import Path
OUT = Path("/content/v5_1_sdqc_results" if os.path.isdir("/content") else "v5_1_sdqc_results")
OUT.mkdir(exist_ok=True, parents=True)

ROWS, SANITY, PROBES = [], [], []
PRIORS = CFG["PRIORS"]
HARDWARE = [("clean", CLEAN)] + TEST_CONDS
if QUICK_TEST:
    HARDWARE = [("clean", CLEAN), TEST_CONDS[0], TEST_CONDS[-1]]
REF_PROBE = dict(REF, shots=CFG["REF_PROBE_SHOTS"])
EST = {"SDQC (probe + EM)": "sdqc", "SDQC-sim (probe + EM)": "sdqc_sim", "EM only": "em", "Physics-informed + EM": "phys"}
t_all = time.time()

for flag in CFG["DATASETS"]:
    D, Y = DATA[flag], DATA[flag]["Y"]
    source_prior = float(Y["cal"].mean())
    for seed in CFG["SEEDS"]:
        t0 = time.time()
        models = {
            "VQC": VQCModel("VQC", D, seed, False),
            "Re-uploading VQC": VQCModel("Re-uploading VQC", D, seed, True),
            "QSVM": QSVMModel("QSVM", D, seed),
            "MLP": ClassicalModel("MLP", D, seed),
            "RBF-SVM": ClassicalModel("RBF-SVM", D, seed),
        }
        for name, m in models.items():
            val_auc = float(roc_auc_score(Y["val"], m.clean("val")))
            SANITY.append(dict(dataset=flag, seed=seed, model=name, val_AUROC=val_auc,
                               test_AUROC=float(roc_auc_score(Y["test"], m.clean("test"))),
                               passed_gate=val_auc >= CFG["AUROC_GATE"]))
            if m.kind != "quantum" or val_auc < CFG["AUROC_GATE"]:
                continue
            rng = np.random.default_rng(100000 + 1000 * seed + sum(map(ord, name)))

            # ---------------- calibration day: hardware in reference state REF ----------------
            fixed = fit_platt(m.observe("cal", REF, rng), Y["cal"])            # calibration-day Platt (labels)
            z_ref = m.observe("cal", REF, rng)                                  # unlabeled stats (moment matching)
            probe_idx = select_probe_indices(len(Y["val"]), CFG["PROBE_N"], 3000 + seed)
            probe_ref_hw = m.observe("val", REF_PROBE, rng)[probe_idx]          # stored hardware reference (no labels)

            # ---------------- comparators that need a noise-free simulator ----------------
            clean_platt = fit_platt(m.clean("cal"), Y["cal"])
            probe_ref_sim = m.clean("val")[probe_idx]
            z_train = [m.observe("cal", c, rng) for c in TRAIN_CONDS]
            noise_cond = fit_noise_conditioned_platt(z_train, TRAIN_CONDS, Y["cal"])
            phys_corr, _ = fit_device_response(m.clean("cal"), z_train, TRAIN_CONDS)

            for hi, (family, hardware) in enumerate(HARDWARE):
                reported_hw = reported(hardware)
                probe_today = m.observe("val", hardware, rng)[probe_idx]        # same probes, today's hardware
                corr_hw, info_hw = fit_probe_response(probe_ref_hw, probe_today, CFG["SDQC_MIN_SLOPE"], CFG["SDQC_MAX_SLOPE"])
                corr_sim, info_sim = fit_probe_response(probe_ref_sim, probe_today, CFG["SDQC_MIN_SLOPE"], CFG["SDQC_MAX_SLOPE"])
                for ref_name, info in [("hardware (calibration day)", info_hw), ("simulator", info_sim)]:
                    PROBES.append(dict(dataset=flag, seed=seed, model=name, hardware_id=hi,
                                       hardware_family=family, reference=ref_name, **info))
                z_target = m.observe("test", hardware, rng)
                oracle = fit_platt(m.observe("cal", hardware, rng), Y["cal"])
                for prior in PRIORS:
                    ids = sample_label_shift(Y["test"], prior, 12345 + hi + seed)
                    if not len(ids):
                        continue
                    z, y = z_target[ids], Y["test"][ids]
                    pi_true = float(y.mean())
                    sh = CFG["SDQC_EM_SHRINK"]
                    p_fixed = fixed(z)
                    p_probe = fixed(corr_hw(z))
                    p_sdqc, est_sdqc = em_prior_with_info(p_probe, source_prior, sh)
                    p_probe_sim = clean_platt(corr_sim(z))
                    p_sdqc_sim, est_sim = em_prior_with_info(p_probe_sim, source_prior, sh)
                    p_phys = clean_platt(phys_corr(z, reported_hw))
                    p_phys_em, est_phys = em_prior_with_info(p_phys, source_prior, sh)
                    p_em_only, est_em = em_prior_with_info(p_fixed, source_prior, sh)
                    p_oracle = oracle(z)
                    est = dict(sdqc=est_sdqc, sdqc_sim=est_sim, em=est_em, phys=est_phys)
                    predictions = {
                        "Fixed Platt": p_fixed,
                        "Oracle Platt (noise only)": p_oracle,
                        "Oracle (noise + true prior)": prior_adjust(p_oracle, source_prior, pi_true),
                        "Noise-conditioned Platt": noise_cond(z, reported_hw),
                        "Physics-informed": p_phys,
                        "Physics-informed + EM": p_phys_em,
                        "Moment matching": fixed(moment_match(z, z_ref.mean(), z_ref.std())),
                        "EM only": p_em_only,
                        "SDQC (probe only)": p_probe,
                        "SDQC (probe + EM)": p_sdqc,
                        "SDQC-sim (probe only)": p_probe_sim,
                        "SDQC-sim (probe + EM)": p_sdqc_sim,
                    }
                    for method, p in predictions.items():
                        ROWS.append(dict(dataset=flag, seed=seed, model=name, hardware_id=hi,
                                         hardware_family=family, p=hardware["p"], g=hardware["g"],
                                         target_prior=str(prior), actual_prior=pi_true,
                                         estimated_prior=est.get(EST.get(method), np.nan),
                                         method=method, **metrics(p, y)))
        print(f"{flag} seed {seed}: done in {time.time() - t0:.0f}s", flush=True)
        pd.DataFrame(ROWS).to_csv(OUT / "metrics_partial.csv", index=False)

results, sanity, probes = pd.DataFrame(ROWS), pd.DataFrame(SANITY), pd.DataFrame(PROBES)
results.to_csv(OUT / "v5_1_metrics.csv", index=False)
sanity.to_csv(OUT / "v5_1_sanity.csv", index=False)
probes.to_csv(OUT / "v5_1_probe_diagnostics.csv", index=False)
with open(OUT / "v5_1_config.json", "w") as f:
    json.dump(dict(CFG=CFG, QUICK_TEST=QUICK_TEST,
                   limitations=["OCTMNIST is CNV-vs-NORMAL only, pool subsampled to OCT_MAX_POOL",
                                "noise is simulated and input-independent",
                                "QSVM noisy kernel is a density-matrix proxy",
                                "EM assumes label shift"]), f, indent=2, default=str)
print(f"Saved {len(results)} rows to {OUT} in {(time.time() - t_all) / 60:.1f} min")

## 9. Results

Primary metric: Brier score (lower is better). Comparisons are paired: same dataset, seed, model, hardware condition and disease rate. Confidence intervals come from a bootstrap over (seed × model × hardware condition) clusters, so repeated evaluations of the same condition are not treated as independent.

In [ ]:
display(sanity.groupby(["dataset", "model"])[["val_AUROC", "test_AUROC", "passed_gate"]].mean().round(3))

SHOW = ["Fixed Platt", "Moment matching", "EM only", "Physics-informed + EM", "SDQC-sim (probe + EM)",
        "SDQC (probe only)", "SDQC (probe + EM)", "Oracle (noise + true prior)"]
results["prior_type"] = np.where(results.target_prior == "source", "same disease rate", "changed disease rate")

print("\nMean Brier score by dataset, hardware family and disease-rate setting")
tab = (results[results.method.isin(SHOW)]
       .groupby(["dataset", "hardware_family", "prior_type", "method"])["Brier"].mean()
       .unstack("method")[SHOW].round(4))
display(tab)
tab.to_csv(OUT / "v5_1_table_brier.csv")

# ---- paired bootstrap: SDQC (probe + EM) vs each comparator ----
keys = ["dataset", "seed", "model", "hardware_id", "target_prior"]
pv = results.pivot_table(index=keys + ["hardware_family", "prior_type"], columns="method", values="Brier").reset_index()
pv["cluster"] = pv.seed.astype(str) + "|" + pv.model + "|" + pv.hardware_id.astype(str)
rb = np.random.default_rng(0)
rows = []
for (ds, fam, pt), g in pv[pv.hardware_family != "clean"].groupby(["dataset", "hardware_family", "prior_type"]):
    clusters = g.cluster.unique()
    for comp in ["Fixed Platt", "Moment matching", "EM only", "Physics-informed + EM", "SDQC-sim (probe + EM)",
                 "SDQC (probe only)", "Oracle (noise + true prior)"]:
        d = (g["SDQC (probe + EM)"] - g[comp]).groupby(g.cluster).mean()
        boots = [d.loc[rb.choice(clusters, len(clusters))].mean() for _ in range(CFG["N_BOOT"])]
        lo, hi = np.percentile(boots, [2.5, 97.5])
        rows.append(dict(dataset=ds, hardware=fam, disease_rate=pt, vs=comp, diff=round(d.mean(), 4),
                         CI95=f"[{lo:.4f}, {hi:.4f}]",
                         SDQC="better" if hi < 0 else ("worse" if lo > 0 else "no clear difference")))
paired = pd.DataFrame(rows)
print("\nSDQC (probe + EM) minus comparator, Brier (negative = SDQC better)")
display(paired)
paired.to_csv(OUT / "v5_1_paired_sdqc.csv", index=False)

# ---- EM disease-rate estimates ----
est = results[results.method.isin(EST) & (results.prior_type == "changed disease rate")].copy()
est["abs_error"] = (est.estimated_prior - est.actual_prior).abs()
print("\nMean absolute error of the estimated disease rate (changed-rate settings)")
display(est.groupby(["dataset", "hardware_family", "method"])["abs_error"].mean().unstack().round(3))

# ---- probe diagnostics: hardware vs simulator reference ----
print("\nProbe fit diagnostics")
display(probes[probes.hardware_family != "clean"].groupby(["dataset", "reference"])[["slope", "probe_rmse"]].mean().round(3))

# ---- figure: the hard case (unreported noise + changed disease rate) ----
FIG = ["Fixed Platt", "Moment matching", "EM only", "Physics-informed + EM", "SDQC (probe + EM)", "Oracle (noise + true prior)"]
hard = results[(results.hardware_family == "unseen: amplitude damping") & (results.prior_type == "changed disease rate")]
dsets = list(hard.dataset.unique())
fig, axes = plt.subplots(1, len(dsets), figsize=(4.6 * len(dsets), 3.8), squeeze=False)
for ax, ds in zip(axes[0], dsets):
    v = hard[hard.dataset == ds].groupby("method")["Brier"].mean().reindex(FIG)
    ax.barh(range(len(FIG)), v.values, color=["#9aa5b1"] * 4 + ["#c2410c", "#334155"])
    ax.set_yticks(range(len(FIG))); ax.set_yticklabels(FIG if ax is axes[0][0] else [""] * len(FIG), fontsize=9)
    ax.invert_yaxis(); ax.set_title(ds, fontsize=11); ax.set_xlabel("Brier score (lower is better)")
fig.suptitle("Unreported noise + changed disease rate", fontsize=12)
plt.tight_layout(); plt.savefig(OUT / "v5_1_hard_case.png", dpi=200); plt.show()

## 10. How to read the results, and next steps

* **SDQC vs SDQC-sim:** the price of removing the simulator. If SDQC is close to SDQC-sim, the method works with hardware-only information, which is the version to report.
* **SDQC vs Oracle (noise + true prior):** how much is left to gain. The oracle uses labels and the true disease rate, so it is a best case.
* **Same vs changed disease rate:** SDQC should match moment matching when the disease rate is unchanged and beat it clearly when the rate changes.
* **EM only:** if it is worse than Fixed Platt under noise, that confirms that hardware must be corrected *before* estimating the disease rate.

**Before a paper:** probe budget (16/32/64/128 probes), input-dependent or coherent noise (where probes may not transfer), drift in the probe reference itself, covariate shift (image corruption, where EM's assumption fails), 5–10 seeds with several data splits, and a device-derived noise model.

In [ ]:
import shutil
archive = shutil.make_archive(str(OUT), "zip", str(OUT))
print("Results archive:", archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    pass
